In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import URL

ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'
PROCESSED_DIR = os.path.join(ROOT, 'data', 'processed')
REPORTS_DIR = os.path.join(ROOT, 'reports')
DB_PATH = os.path.join(PROCESSED_DIR, 'bluestock_dw.db')

engine = create_engine(URL.create('sqlite', database=DB_PATH))

@event.listens_for(engine, 'connect')
def turn_on_foreign_keys(dbapi_connection, connection_record):
    cursor = dbapi_connection.cursor()
    cursor.execute('PRAGMA foreign_keys = ON')
    cursor.close()

In [3]:
bounds = pd.read_sql_query("""
SELECT n.amfi_code,
       MIN(d.full_date) AS first_date,
       MAX(d.full_date) AS last_date,
       COUNT(*)         AS nav_rows
FROM fact_nav AS n
JOIN dim_date AS d ON d.date_key = n.date_key
GROUP BY n.amfi_code
""", engine)

bounds['first_date'] = pd.to_datetime(bounds['first_date'])
bounds['last_date'] = pd.to_datetime(bounds['last_date'])

END_DATE = bounds['last_date'].max()

print('Funds:', len(bounds))
print('End date:', END_DATE.date())
print('Oldest first date:', bounds['first_date'].min().date(), '| newest first date:', bounds['first_date'].max().date())

stale = bounds[bounds['last_date'] < END_DATE]
print('Funds whose last NAV is before the end date:', len(stale)) 

Funds: 40
End date: 2026-05-29
Oldest first date: 2022-01-03 | newest first date: 2022-01-03
Funds whose last NAV is before the end date: 0


In [4]:
nav = pd.read_sql_query("""
SELECT n.amfi_code, d.full_date, n.nav, n.is_filled
FROM fact_nav AS n
JOIN dim_date AS d ON d.date_key = n.date_key
""", engine)

nav['date'] = pd.to_datetime(nav['full_date'])

nav_wide = nav.pivot(index='date', columns='amfi_code', values='nav').sort_index()
filled_wide = nav.pivot(index='date', columns='amfi_code', values='is_filled').sort_index()

print(nav_wide.shape)
print(nav_wide.index.min().date(), 'to', nav_wide.index.max().date())

(1608, 40)
2022-01-03 to 2026-05-29


In [5]:
def nav_on(date):
    return nav_wide.reindex([date]).iloc[0]


def filled_on(date):
    return filled_wide.reindex([date]).iloc[0]


nav_end = nav_on(END_DATE)

start_dates = {}
for n in [1, 3, 5]:
    start_dates[n] = END_DATE - pd.DateOffset(years=n)
    print(n, 'year start date:', start_dates[n].date(), '| weekday:', start_dates[n].day_name())

print('Funds with an end NAV:', nav_end.notnull().sum())

1 year start date: 2025-05-29 | weekday: Thursday
3 year start date: 2023-05-29 | weekday: Monday
5 year start date: 2021-05-29 | weekday: Saturday
Funds with an end NAV: 40


In [6]:
results = pd.DataFrame(index=nav_wide.columns)
results.index.name = 'amfi_code'
results['nav_end'] = nav_end

for n in [1, 3, 5]:
    nav_start = nav_on(start_dates[n])
    results[f'nav_start_{n}yr'] = nav_start
    results[f'total_{n}yr_pct'] = (nav_end / nav_start - 1) * 100
    results[f'cagr_{n}yr_pct'] = ((nav_end / nav_start) ** (1 / n) - 1) * 100

for n in [1, 3, 5]:
    print(n, 'year: funds with a CAGR:', results[f'cagr_{n}yr_pct'].notnull().sum(), 'of', len(results))

results[['cagr_1yr_pct', 'cagr_3yr_pct', 'cagr_5yr_pct']].describe().round(2)

1 year: funds with a CAGR: 40 of 40
3 year: funds with a CAGR: 40 of 40
5 year: funds with a CAGR: 0 of 40


,cagr_1yr_pct,cagr_3yr_pct,cagr_5yr_pct
count,40.00,40.00,0.0
mean,19.43,16.41,NaN
std,22.91,12.21,NaN
min,-42.80,-11.71,NaN
25%,7.38,6.60,NaN
50%,17.47,18.23,NaN
75%,27.16,26.90,NaN
max,82.78,35.11,NaN


In [7]:
# Check A
same = np.allclose(results['cagr_1yr_pct'], results['total_1yr_pct'], equal_nan=True)
print('A. 1-year CAGR equals 1-year total return:', same)

# Check B
for n in [1, 3, 5]:
    values = results[f'cagr_{n}yr_pct'].dropna()
    print(f'B. {n}-year CAGR above -100 for all funds:', bool((values > -100).all()))

# Check C
code = int(results['cagr_3yr_pct'].dropna().index[0])
start_key = int(start_dates[3].strftime('%Y%m%d'))
end_key = int(END_DATE.strftime('%Y%m%d'))

spot = pd.read_sql_query(
    text('SELECT date_key, nav FROM fact_nav WHERE amfi_code = :code AND date_key IN (:a, :b)'),
    engine, params={'code': code, 'a': start_key, 'b': end_key})

nav_s = spot.loc[spot['date_key'] == start_key, 'nav'].iloc[0]
nav_e = spot.loc[spot['date_key'] == end_key, 'nav'].iloc[0]
manual = ((nav_e / nav_s) ** (1 / 3) - 1) * 100

print('C. Fund', code, '| by hand:', round(manual, 4), '| in the table:', round(results.loc[code, 'cagr_3yr_pct'], 4))
print('C. Same value:', np.isclose(manual, results.loc[code, 'cagr_3yr_pct']))

A. 1-year CAGR equals 1-year total return: True
B. 1-year CAGR above -100 for all funds: True
B. 3-year CAGR above -100 for all funds: True
B. 5-year CAGR above -100 for all funds: True
C. Fund 100016 | by hand: 1.2926 | in the table: 1.2926
C. Same value: True


In [13]:
lines = [
    '# CAGR summary',
    '',
    f'- Formula: CAGR = (NAV_end / NAV_start) ^ (1 / n) - 1, for n = 1, 3, and 5 years. Shown in percent.',
    f'- End date: {END_DATE.date()}. Start dates: ' + ', '.join(f'{n} yr {start_dates[n].date()}' for n in [1, 3, 5]) + '.',
    '- NAV on a weekend or holiday is the last known NAV (filled rows from Task 8).',
   
   ]

summary_text = '\n'.join(lines)

with open(os.path.join(REPORTS_DIR, 'cagr_summary.md'), 'w', encoding='utf-8') as f:
    f.write(summary_text)

print(summary_text)

# CAGR summary

- Formula: CAGR = (NAV_end / NAV_start) ^ (1 / n) - 1, for n = 1, 3, and 5 years. Shown in percent.
- End date: 2026-05-29. Start dates: 1 yr 2025-05-29, 3 yr 2023-05-29, 5 yr 2021-05-29.
- NAV on a weekend or holiday is the last known NAV (filled rows from Task 8).
